# **Azure Cloud CPU Data Processing with Polars**
## **Notebook Aim**
This notebook aims to efficiently process and analyze Azure Cloud CPU usage data using the **Polars** library. It focuses on:
- Reading and concatenating multiple compressed `.csv.gz` files quickly.
- Filtering virtual machines (`vmid`) that have at least **29 consecutive days** of recorded data.
- Handling anonymized `vmid` values while ensuring proper data formatting.
- Removing escape characters from the dataset for clean processing.
- Storing and managing the filtered dataset in an optimized manner.

## **Tools & Libraries Used**
- `Polars`: A fast DataFrame library for efficient data handling instead of `Pandas`


## **Dataset Information**
- **Format**: Compressed CSV files (`.csv.gz`)
- **Key Columns**: `vmid`, timestamps, CPU utilization metrics.
- **Objective**: Identify and retain only `vmid` values with at least **29 consecutive days** of data.

## **Expected Outcome**
- A clean, filtered dataset with long-term `vmid` data.
- Faster and optimized data operations using `Polars`.
- Readable and structured data without escape characters.

---


In [ ]:
import polars as pl
import glob
import os
import re

#import warnings
#warnings.filterwarnings("ignore", category=DeprecationWarning)

In [ ]:
def sanitize_filename(vmid):
    return re.sub(r'[<>:"/\\|?*]', '_', vmid)

In [ ]:
pwd

'/home/mehryar'

In [ ]:
%%time
# Define output directory
output_dir = "/home/mehryar/VM_30_days"
os.makedirs(output_dir, exist_ok=True)  # Create the output directory if it doesn't exist

# Search for all csv.gz files in the folder
path = "/home/mehryar/vm_cpu_readings/data/*.csv.gz"
#files = glob.glob(path)
files = sorted(glob.glob(path))
lazy_dfs = []

column_names = ['timestamp', 'vmid', 'mincpu', 'maxcpu', 'avgcpu']

for file in files:
    df = pl.read_csv(file, has_header=False, new_columns=column_names)
    lazy_df = df.lazy()
    lazy_dfs.append(lazy_df)
    print(f'Read: {file}')

# Combine all dataframes into a single dataframe
combined_lazy_df = pl.concat(lazy_dfs)

Read: /home/mehryar/vm_cpu_readings/data/vm_cpu_readings-file-164-of-195.csv.gz
Read: /home/mehryar/vm_cpu_readings/data/vm_cpu_readings-file-62-of-195.csv.gz
Read: /home/mehryar/vm_cpu_readings/data/vm_cpu_readings-file-86-of-195.csv.gz
Read: /home/mehryar/vm_cpu_readings/data/vm_cpu_readings-file-41-of-195.csv.gz
Read: /home/mehryar/vm_cpu_readings/data/vm_cpu_readings-file-56-of-195.csv.gz
Read: /home/mehryar/vm_cpu_readings/data/vm_cpu_readings-file-180-of-195.csv.gz
Read: /home/mehryar/vm_cpu_readings/data/vm_cpu_readings-file-33-of-195.csv.gz
Read: /home/mehryar/vm_cpu_readings/data/vm_cpu_readings-file-78-of-195.csv.gz
Read: /home/mehryar/vm_cpu_readings/data/vm_cpu_readings-file-80-of-195.csv.gz
Read: /home/mehryar/vm_cpu_readings/data/vm_cpu_readings-file-21-of-195.csv.gz
Read: /home/mehryar/vm_cpu_readings/data/vm_cpu_readings-file-124-of-195.csv.gz
Read: /home/mehryar/vm_cpu_readings/data/vm_cpu_readings-file-101-of-195.csv.gz
Read: /home/mehryar/vm_cpu_readings/data/vm_cpu_

In [ ]:
#print("Combined Columns:", combined_lazy_df.columns)
print("Combined Columns:", combined_lazy_df.collect_schema().names())

Combined Columns: ['timestamp', 'vmid', 'mincpu', 'maxcpu', 'avgcpu']


In [ ]:
%%time
new_column_names = column_names

if len(combined_lazy_df.collect_schema().names()) == len(new_column_names):
    combined_lazy_df = combined_lazy_df.rename(
        {old_name: new_name for old_name, new_name in zip(combined_lazy_df.collect_schema().names(), new_column_names)}
    )
##avoid DeprecationWarning combined_lazy_df.columns ---> combined_lazy_df.collect_schema().names()
#if all(col in combined_lazy_df.columns for col in ['timestamp', 'vmid', 'mincpu', 'maxcpu', 'avgcpu']):
if all(col in combined_lazy_df.collect_schema().names() for col in ['timestamp', 'vmid', 'mincpu', 'maxcpu', 'avgcpu']):
    # Get the count of rows for each vmid
    #vmid_counts = combined_lazy_df.group_by("vmid").count().collect()
    ##avoid DeprecationWarning lf.group_by("a").count().collect() ---> lf.group_by("a").len().collect()
    vmid_counts = combined_lazy_df.group_by("vmid").len().collect()
    # print(vmid_counts)
    # Filter vmids with more than 8064 rows
    #valid_vmids = vmid_counts.filter(pl.col("count") > 8350)["vmid"].to_list()    #20*288=5760
    #print("Unique VMIDs with more than 8064 rows:", len(valid_vmids))             #180,957
    #valid_vmids = vmid_counts.filter(pl.col("count") >= 8640)["vmid"].to_list()   #30*288=8640
    #print("Unique VMIDs contain 8640 rows:", len(valid_vmids))                    #0,000
    valid_vmids = vmid_counts.filter(pl.col("len") >= 8352)["vmid"].to_list()      #29*288=8352  pl.col("count") ---> pl.col("len")
    print("Unique VMIDs contain 8352 rows:", len(valid_vmids))                     #179,393
    #valid_vmids = vmid_counts.filter(pl.col("count") >= 8064)["vmid"].to_list()   #28*288=8064
    #print("Unique VMIDs contain 8064 rows:", len(valid_vmids))                    #0,000

    # Convert the valid_vmids list to a DataFrame
    valid_vmids_df = pl.DataFrame({"vmid": valid_vmids})
    # Define the name of the output file
    output_file = os.path.join(output_dir, "valid_vmids.csv")
    # Save DataFrame as CSV
    valid_vmids_df.write_csv(output_file)

    print(f"Valid VMIDs have been saved to: {output_file}")

Unique VMIDs contain 8352 rows: 179393
Saving data for vmid Ke4Jm9k2LMSs+EoElqWVvIirz37n37bNs8rEyP1KJKm3Y_+AuaIqSSgcYC3kfjKl in /home/mehryar/VM_30_days/vmid_Ke4Jm9k2LMSs+EoElqWVvIirz37n37bNs8rEyP1KJKm3Y_+AuaIqSSgcYC3kfjKl.csv...
Saved data for vmid Ke4Jm9k2LMSs+EoElqWVvIirz37n37bNs8rEyP1KJKm3Y_+AuaIqSSgcYC3kfjKl in /home/mehryar/VM_30_days/vmid_Ke4Jm9k2LMSs+EoElqWVvIirz37n37bNs8rEyP1KJKm3Y_+AuaIqSSgcYC3kfjKl.csv
Saving data for vmid iB2zeshbvBnDscNx8QJs00hQNsNe4b1sl2XON+dbWwJ03avljTcfPPEML7x3wZms in /home/mehryar/VM_30_days/vmid_iB2zeshbvBnDscNx8QJs00hQNsNe4b1sl2XON+dbWwJ03avljTcfPPEML7x3wZms.csv...
Saved data for vmid iB2zeshbvBnDscNx8QJs00hQNsNe4b1sl2XON+dbWwJ03avljTcfPPEML7x3wZms in /home/mehryar/VM_30_days/vmid_iB2zeshbvBnDscNx8QJs00hQNsNe4b1sl2XON+dbWwJ03avljTcfPPEML7x3wZms.csv
Saving data for vmid VgCT5HCLBDPSfFiFFUchjvMzloz1miQOhDLcPpQ77ioljLts4fzCiBaZfnyLo+Ng in /home/mehryar/VM_30_days/vmid_VgCT5HCLBDPSfFiFFUchjvMzloz1miQOhDLcPpQ77ioljLts4fzCiBaZfnyLo+Ng.csv...
Saved data for

KeyboardInterrupt: 